<a href="https://colab.research.google.com/github/lidchen/ToyTransformer/blob/main/transformer_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import torch

ds = load_dataset("roneneldan/TinyStories")
train_ds = ds["train"]
text = ""
for i in range(5000):
    text += train_ds[i]["text"]
chars = sorted(list(set(text)))

stoi = {
    ch:i
    for i,ch in enumerate(chars)
}
itos = {
    i:ch
    for ch,i in stoi.items()
}
all_text = ""

for i in range(5000):
    all_text += train_ds[i]["text"]

data = torch.tensor(
  [stoi[c] for c in all_text],
  dtype=torch.long
)
def get_batch(B, T):
    ix = torch.randint(
        0,
        len(data) - T - 1,
        (B,)
    )

    x = torch.stack([
        data[i:i+T]
        for i in ix
    ])
    y = torch.stack([
        data[i+1:i+T+1]
        for i in ix
    ])
    return x, y

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import torch

block_size = 128

ds = load_dataset("roneneldan/TinyStories")
train_ds = ds["train"]

# build vocab
text = ""
for i in range(5000):
    text += train_ds[i]["text"]

chars = sorted(list(set(text)))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

class TinyStoriesDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = []

        for item in hf_dataset:
            txt = item["text"]

            # encode
            tokens = [stoi[c] for c in txt if c in stoi]

            # make chunks
            for i in range(0, len(tokens) - block_size):
                x = tokens[i:i+block_size]
                y = tokens[i+1:i+block_size+1]

                self.data.append((
                    torch.tensor(x),
                    torch.tensor(y)
                ))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


# loader = DataLoader(
#     dataset,
#     batch_size=64,
#     shuffle=True
# )


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [ ]:
dataset = TinyStoriesDataset(train_ds.select(range(1000)))

params = {'batch_size': 64,
          'shuffle': True,
          'num_workers': 2}

training_generator = torch.utils.data.DataLoader(dataset, **params)

In [ ]:
# Multiple layers
import torch
import torch.nn as nn
import torch.nn.functional as F
class TransformerBlock(nn.Module):
  def __init__(self, embed_dim, head_num, head_size, block_size):
    super().__init__()
    self.ln1 = nn.LayerNorm(embed_dim)
    self.ln2 = nn.LayerNorm(embed_dim)
    self.query = nn.Linear(embed_dim, head_num * head_size, bias=False)
    self.key   = nn.Linear(embed_dim, head_num * head_size, bias=False)
    self.value = nn.Linear(embed_dim, head_num * head_size, bias=False)
    self.proj  = nn.Linear(embed_dim, embed_dim)
    self.ffn   = nn.Sequential(
      nn.Linear(embed_dim, 4 * embed_dim),
      nn.GELU(),
      nn.Linear(4 * embed_dim, embed_dim)
    )
    self.head_num  = head_num
    self.head_size = head_size
    self.register_buffer(
      "tril",
      torch.tril(torch.ones(block_size, block_size))
    )

  def forward(self, x):
    B, T, C = x.shape

    # Pre-norm attention
    x_norm = self.ln1(x)
    q = self.query(x_norm).view(B, T, self.head_num, self.head_size).transpose(1, 2)
    k = self.key(x_norm).view(B, T, self.head_num, self.head_size).transpose(1, 2)
    v = self.value(x_norm).view(B, T, self.head_num, self.head_size).transpose(1, 2)

    w = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)
    w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    w = F.softmax(w, dim=-1)

    out = (w @ v).transpose(1, 2).contiguous().view(B, T, -1)
    x = x + self.proj(out)

    # Pre-norm FFN
    x = x + self.ffn(self.ln2(x))
    return x

class multiLayerTransformer(nn.Module):
  def __init__(self, vocab_size, block_size, embed_dim, head_num, head_size, num_layers):
    super().__init__()
    self.block_size = block_size
    self.token_embedding    = nn.Embedding(vocab_size, embed_dim)
    self.position_embedding = nn.Embedding(block_size, embed_dim)

    self.blocks = nn.Sequential(*[
        TransformerBlock(embed_dim, head_num, head_size, block_size)
        for _ in range(num_layers)
    ])

    self.lnf     = nn.LayerNorm(embed_dim)
    self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
    self.lm_head.weight = self.token_embedding.weight  # weight tying

  def forward(self, idx, targets=None):
    B, T = idx.shape
    x = self.token_embedding(idx) + self.position_embedding(
      torch.arange(T, device=idx.device)
    )

    x = self.blocks(x)      # passes through all N transformer blocks
    x = self.lnf(x)
    logits = self.lm_head(x)

    loss = None
    if targets is not None:
      loss = F.cross_entropy(
        logits.view(-1, logits.shape[-1]),
        targets.view(-1)
      )
    return logits, loss

  @torch.no_grad()
  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -self.block_size:]   # crop to block_size
      logits, _ = self(idx_cond)
      probs = F.softmax(logits[:, -1, :], dim=-1)
      next_idx = torch.multinomial(probs, num_samples=1)
      idx = torch.cat([idx, next_idx], dim=1)
    return idx

In [ ]:
# train a new model
from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = multiLayerTransformer(
  vocab_size=len(chars),
  block_size=128,
  embed_dim=512,
  head_num=8,
  head_size=64,
  num_layers=4
)
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=3e-4
)

for step in range(5000):
  idx, targets = get_batch(64, model.block_size)
  idx = idx.to(device)
  targets = targets.to(device)
  logits, loss = model(idx, targets)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if step % 1000 == 0:
    print(loss.item())

# train a new model
torch.save(model.state_dict(), "/content/drive/MyDrive/multi_layer_model.pt")

In [ ]:
# load model
from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = multiLayerTransformer(
  vocab_size=len(chars),
  block_size=128,
  embed_dim=512,
  head_num=8,
  head_size=64,
  num_layers=4
)
model = model.to(device)

model.load_state_dict(
  torch.load(
    "/content/drive/MyDrive/multi_layer_model.pt",
    map_location=device
  )
)
print("model loaded")


Mounted at /content/drive
cuda
model loaded


In [ ]:
# train a new model with data loader
from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = multiLayerTransformer(
  vocab_size=len(chars),
  block_size=128,
  embed_dim=512,
  head_num=8,
  head_size=64,
  num_layers=4
)
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=3e-4
)

# Loop over epochs
for epoch in range(max_epochs):
  # Training
  for idx, targets in training_generator:
    # Transfer to GPU
    idx, targets = idx.to(device), targets.to(device)
    logits, loss = model(idx, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
      print(loss.item())

# train a new model
torch.save(model.state_dict(), "/content/drive/MyDrive/multi_layer_model.pt")


In [ ]:
# Continue training

model.train()
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=1e-3
)

for step in range(5000):
  idx, targets = get_batch(64, model.block_size)
  idx = idx.to(device)
  targets = targets.to(device)
  logits, loss = model(idx, targets)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if step % 1000 == 0:
    print(loss.item())
    torch.save(model.state_dict(), "/content/drive/MyDrive/multi_layer_model.pt")

# torch.save( model.state_dict(), "model.pt" )
torch.save(model.state_dict(), "/content/drive/MyDrive/multi_layer_model.pt")


0.9160068035125732
0.9165351390838623


KeyboardInterrupt: 

In [ ]:
# Continue training with dataloader

model.train()
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=3e-4
)

max_epochs = 100
# Loop over epochs

step = 0
for epoch in range(max_epochs):
  # Training
  for idx, targets in training_generator:
    # Transfer to GPU
    idx, targets = idx.to(device), targets.to(device)
    logits, loss = model(idx, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
      print(loss.item())
      torch.save(model.state_dict(), "/content/drive/MyDrive/multi_layer_model.pt")
    step += 1

# torch.save( model.state_dict(), "model.pt" )
torch.save(model.state_dict(), "/content/drive/MyDrive/multi_layer_model.pt")


0.10604792833328247
0.10744068771600723
0.10931291431188583
0.10717505216598511
0.10960856080055237
0.11056971549987793
0.1063089445233345


In [ ]:
# generate
model.eval()

start = torch.tensor([[stoi["T"]]])
start = start.to(device)
out = model.generate(
  start,
  max_new_tokens=1000
)
print(out)

decoded = ''.join(
  [itos[i.item()] for i in out[0]]
)
print(decoded)

tensor([[41, 55, 56,  ..., 62, 60,  1]], device='cuda:0')
This time they were pushing back, inviting her closer. As the girl looked closer, she saw a crack in the window and she could now hear more clearly the voice she had heard earlier. "Come here," it said. 

The girl took a deep breath, trembling with anticipation and excitement. She stepped closer and the curtain parted. Behind it was a secret garden. Everything was clean and vibrant and the girl felt more alive than she had ever felt before. She was finally on an adventure of a big hat and a pet dog. 

Sophie took the edge of the pool too. She climbed up the ladder and slid down the slide. It was so much fun!

After playing for a while, Lily's friend Jack accidentally knocked over a pot. It broke into many pieces. Jack was scared that he would get in trouble, but Lily told him not to worry. She said it was just a harmless pot and thing to do. So he looked around the room. He saw that there were a few pictures on the walls. 

John